# Diabetes Progression Prediction with Interpretable Linear Regression

## Business use case

An interpretable regression model can help quantify how patient characteristics relate to future disease progression and can serve as a transparent baseline before moving to more complex models.

## Objective

Using the NCSU diabetes dataset, the notebook predicts the one-year progression target `Y` with ordinary least squares. It first fits a full model, then reduces the specification to statistically significant predictors and evaluates that reduced model on a holdout set.

## Result in context

The full training model has **R² = 0.524** and an **out-of-sample R² = 0.477**. The reduced model achieves an **R² = 0.470** and an **out-of-sample R² of 0.485**, showing that a smaller, more interpretable feature set is useful and avoids overfitting of the model.

## About the dataset

Ten baseline variables, age, sex, body mass index, average blood pressure, and six blood serum measurements were obtained for each of n = 442 diabetes patients, as well as the response of interest, a quantitative measure of disease progression one year after baseline.

###Data Set Characteristics:
- Number of Instances: 442
- Number of Attributes: First 10 columns are numeric predictive values
- Target: Column 11 is a quantitative measure of disease progression one year after baseline
###Attribute Information:

- age: age in years
- sex: sex
- bmi: body mass index
- bp: average blood pressure
- s1: tc, total serum cholesterol
- s2: ldl, low-density lipoproteins
- s3: hdl, high-density lipoproteins
- s4: tch, total cholesterol / HDL
- s5: ltg, possibly log of serum triglycerides level
- s6: glu, blood sugar level

Note: Each of the ten feature variables was mean-centered and scaled by the standard deviation multiplied by the number of samples in the standardized version of this dataset (i.e., the sum of squares of each column equals 1). However, the features in this NCSU file are presented in their original, unstandardized form—unlike the version provided in scikit-learn.

- Source URL https://www4.stat.ncsu.edu/~boos/var.select/diabetes.html
- Data URL: https://www4.stat.ncsu.edu/~boos/var.select/diabetes.tab.txt

Note: The Data URL mentioned-above is obtained from the source URL. The source URL provides detailed information about the dataset, variables and also reference links including the dataset link. For more information see: Bradley Efron, Trevor Hastie, Iain Johnstone and Robert Tibshirani (2004) "Least Angle Regression," Annals of Statistics (with discussion), 407-499. (https://web.stanford.edu/~hastie/Papers/LARS/LeastAngle_2002.pdf)

## Step 1 — Load the analysis stack and clinical data

Pandas and NumPy handle the dataset, statsmodels provides interpretable OLS output, and scikit-learn creates the holdout split. The public dataset includes demographic and physiological predictors together with the progression target.


In [ ]:
### import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.formula.api import ols
from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn import datasets

In [ ]:
df = pd.read_csv("https://www4.stat.ncsu.edu/~boos/var.select/diabetes.tab.txt", sep="\t")
print(df.head())

   AGE  SEX   BMI     BP   S1     S2    S3   S4      S5  S6    Y
0   59    2  32.1  101.0  157   93.2  38.0  4.0  4.8598  87  151
1   48    1  21.6   87.0  183  103.2  70.0  3.0  3.8918  69   75
2   72    2  30.5   93.0  156   93.6  41.0  4.0  4.6728  85  141
3   24    1  25.3   84.0  198  131.4  40.0  5.0  4.8903  89  206
4   50    1  23.0  101.0  192  125.4  52.0  4.0  4.2905  80  135


## Step 2 — Inspect data types and encode categorical information

The dataset schema is reviewed before modeling. `SEX` is treated as categorical so the regression coefficient is interpreted relative to a reference category rather than as a continuous numeric effect.


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AGE     442 non-null    int64  
 1   SEX     442 non-null    int64  
 2   BMI     442 non-null    float64
 3   BP      442 non-null    float64
 4   S1      442 non-null    int64  
 5   S2      442 non-null    float64
 6   S3      442 non-null    float64
 7   S4      442 non-null    float64
 8   S5      442 non-null    float64
 9   S6      442 non-null    int64  
 10  Y       442 non-null    int64  
dtypes: float64(6), int64(5)
memory usage: 38.1 KB


In [ ]:
categorical_variables = ['SEX']
df[categorical_variables] = df[categorical_variables].astype('category')
print(df[categorical_variables])

    SEX
0     2
1     1
2     2
3     1
4     1
..   ..
437   2
438   2
439   2
440   1
441   1

[442 rows x 1 columns]


In [ ]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 442 entries, 0 to 441
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   AGE     442 non-null    int64   
 1   SEX     442 non-null    category
 2   BMI     442 non-null    float64 
 3   BP      442 non-null    float64 
 4   S1      442 non-null    int64   
 5   S2      442 non-null    float64 
 6   S3      442 non-null    float64 
 7   S4      442 non-null    float64 
 8   S5      442 non-null    float64 
 9   S6      442 non-null    int64   
 10  Y       442 non-null    int64   
dtypes: category(1), float64(6), int64(4)
memory usage: 35.2 KB


## Step 3 — Review descriptive statistics

Summary statistics provide a basic quality check and help identify the scale and spread of the candidate predictors before the regression is specified.


In [ ]:
dfDescription = df.describe(include='all')
print (dfDescription)

               AGE    SEX         BMI          BP          S1          S2  \
count   442.000000  442.0  442.000000  442.000000  442.000000  442.000000   
unique         NaN    2.0         NaN         NaN         NaN         NaN   
top            NaN    1.0         NaN         NaN         NaN         NaN   
freq           NaN  235.0         NaN         NaN         NaN         NaN   
mean     48.518100    NaN   26.375792   94.647014  189.140271  115.439140   
std      13.109028    NaN    4.418122   13.831283   34.608052   30.413081   
min      19.000000    NaN   18.000000   62.000000   97.000000   41.600000   
25%      38.250000    NaN   23.200000   84.000000  164.250000   96.050000   
50%      50.000000    NaN   25.700000   93.000000  186.000000  113.000000   
75%      59.000000    NaN   29.275000  105.000000  209.750000  134.500000   
max      79.000000    NaN   42.200000  133.000000  301.000000  242.400000   

                S3          S4          S5          S6           Y  
count 

In [ ]:
df[['AGE', 'BMI', 'BP', 'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'Y']].corr()

,AGE,BMI,BP,S1,S2,S3,S4,S5,S6,Y
AGE,1.000000,0.185085,0.335428,0.260061,0.219243,-0.075181,0.203841,0.270774,0.301731,0.187889
BMI,0.185085,1.000000,0.395411,0.249777,0.261170,-0.366811,0.413807,0.446157,0.388680,0.586450
BP,0.335428,0.395411,1.000000,0.242464,0.185548,-0.178762,0.257650,0.393480,0.390430,0.441482
S1,0.260061,0.249777,0.242464,1.000000,0.896663,0.051519,0.542207,0.515503,0.325717,0.212022
S2,0.219243,0.261170,0.185548,0.896663,1.000000,-0.196455,0.659817,0.318357,0.290600,0.174054
S3,-0.075181,-0.366811,-0.178762,0.051519,-0.196455,1.000000,-0.738493,-0.398577,-0.273697,-0.394789
S4,0.203841,0.413807,0.257650,0.542207,0.659817,-0.738493,1.000000,0.617859,0.417212,0.430453
S5,0.270774,0.446157,0.393480,0.515503,0.318357,-0.398577,0.617859,1.000000,0.464669,0.565883
S6,0.301731,0.388680,0.390430,0.325717,0.290600,-0.273697,0.417212,0.464669,1.000000,0.382483
Y,0.187889,0.586450,0.441482,0.212022,0.174054,-0.394789,0.430453,0.565883,0.382483,1.000000


## Step 4 — Create training and test samples

A holdout set is created before fitting the model. This separates model specification from final generalization testing.


In [ ]:
df_train,df_test = train_test_split(df,test_size = 0.3,random_state=42)

## Step 5 — Fit the full OLS model

The first regression includes all available predictors. The statsmodels summary exposes coefficients, uncertainty, p-values, R², and diagnostics, making it useful for both prediction and interpretation. When looking at the OSR2, it has decreased compared to the train prediction, which is a clear sign of overfitting.


In [ ]:
est_train = ols(formula = "Y ~ AGE + SEX + BMI + BP + S1 + S2 + S3 + S4 + S5 + S6", data = df_train).fit()
print(est_train.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.524
Model:                            OLS   Adj. R-squared:                  0.508
Method:                 Least Squares   F-statistic:                     32.86
Date:                Mon, 14 Sep 2026   Prob (F-statistic):           1.37e-42
Time:                        21:24:38   Log-Likelihood:                -1671.5
No. Observations:                 309   AIC:                             3365.
Df Residuals:                     298   BIC:                             3406.
Df Model:                          10                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -341.2349     78.615     -4.341      0.0

In [ ]:
test_pred = est_train.predict(df_test)
r2 = r2_score(df_test['Y'],test_pred)
print('OOS R-squared: '+ str(r2))

OOS R-squared: 0.4772897164322617


## Step 6 — Build a reduced, interpretable specification

The notebook refits the model using the predictors retained from significance review (P>|t| > 0.005). It will be done by iterations, eliminating one variable each time until the best model appears.


In [ ]:
est_train_significant = ols(formula = "Y ~ SEX + BMI + BP + S1 + S3 + S5 + S6", data = df_train).fit()
print(est_train_significant.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.519
Model:                            OLS   Adj. R-squared:                  0.507
Method:                 Least Squares   F-statistic:                     46.33
Date:                Mon, 14 Sep 2026   Prob (F-statistic):           2.74e-44
Time:                        21:24:38   Log-Likelihood:                -1673.3
No. Observations:                 309   AIC:                             3363.
Df Residuals:                     301   BIC:                             3393.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -233.1791     44.470     -5.244      0.0

In [ ]:
test_pred = est_train_significant.predict(df_test)
r2 = r2_score(df_test['Y'],test_pred)
print('OOS R-squared: '+ str(r2))

OOS R-squared: 0.4845193830465294


In [ ]:
est_train_significant = ols(formula = "Y ~ SEX + BMI + BP + S1 + S3 + S5", data = df_train).fit()
print(est_train_significant.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.518
Model:                            OLS   Adj. R-squared:                  0.508
Method:                 Least Squares   F-statistic:                     54.05
Date:                Mon, 14 Sep 2026   Prob (F-statistic):           4.65e-45
Time:                        21:24:38   Log-Likelihood:                -1673.6
No. Observations:                 309   AIC:                             3361.
Df Residuals:                     302   BIC:                             3387.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -222.5572     42.032     -5.295      0.0

In [ ]:
test_pred = est_train_significant.predict(df_test)
r2 = r2_score(df_test['Y'],test_pred)
print('OOS R-squared: '+ str(r2))

OOS R-squared: 0.4816385561537895


In [ ]:
est_train_significant = ols(formula = "Y ~ SEX + BMI + BP + S3 + S5", data = df_train).fit()
print(est_train_significant.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.513
Model:                            OLS   Adj. R-squared:                  0.505
Method:                 Least Squares   F-statistic:                     63.80
Date:                Mon, 14 Sep 2026   Prob (F-statistic):           2.50e-45
Time:                        21:24:38   Log-Likelihood:                -1675.2
No. Observations:                 309   AIC:                             3362.
Df Residuals:                     303   BIC:                             3385.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -209.1199     41.470     -5.043      0.0

In [ ]:
test_pred = est_train_significant.predict(df_test)
r2 = r2_score(df_test['Y'],test_pred)
print('OOS R-squared: '+ str(r2))

OOS R-squared: 0.47771009386172447


## Step 7 — Obtaining the best model possible

After checking correlations and variable importance, the notebook will now reduce one varialbe of the 5 remaining to see if there is a better model with 4 variables. It will serach which ones doesn't have a large condition number.


In [ ]:
est_train_significant = ols(formula = "Y ~ BMI + BP + S3 + S5", data = df_train).fit()
print(est_train_significant.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.494
Model:                            OLS   Adj. R-squared:                  0.487
Method:                 Least Squares   F-statistic:                     74.07
Date:                Mon, 14 Sep 2026   Prob (F-statistic):           9.25e-44
Time:                        21:24:38   Log-Likelihood:                -1681.2
No. Observations:                 309   AIC:                             3372.
Df Residuals:                     304   BIC:                             3391.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -237.3403     41.392     -5.734      0.0

In [ ]:
test_pred = est_train_significant.predict(df_test)
r2 = r2_score(df_test['Y'],test_pred)
print('OOS R-squared: '+ str(r2))

OOS R-squared: 0.4632833743936884


In [ ]:
est_train_significant = ols(formula = "Y ~ SEX + BP + S3 + S5", data = df_train).fit()
print(est_train_significant.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.433
Model:                            OLS   Adj. R-squared:                  0.426
Method:                 Least Squares   F-statistic:                     58.10
Date:                Mon, 14 Sep 2026   Prob (F-statistic):           2.18e-36
Time:                        21:24:38   Log-Likelihood:                -1698.6
No. Observations:                 309   AIC:                             3407.
Df Residuals:                     304   BIC:                             3426.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -126.5457     42.832     -2.954      0.0

In [ ]:
test_pred = est_train_significant.predict(df_test)
r2 = r2_score(df_test['Y'],test_pred)
print('OOS R-squared: '+ str(r2))

OOS R-squared: 0.4237132061509532


In [ ]:
est_train_significant = ols(formula = "Y ~ SEX + BMI + S3 + S5", data = df_train).fit()

In [ ]:
test_pred = est_train_significant.predict(df_test)
r2 = r2_score(df_test['Y'],test_pred)
print('OOS R-squared: '+ str(r2))

OOS R-squared: 0.48511853284845097


In [ ]:
est_train_significant = ols(formula = "Y ~ SEX + BMI + BP + S5", data = df_train).fit()
print(est_train_significant.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.481
Model:                            OLS   Adj. R-squared:                  0.474
Method:                 Least Squares   F-statistic:                     70.44
Date:                Mon, 14 Sep 2026   Prob (F-statistic):           3.76e-42
Time:                        21:24:38   Log-Likelihood:                -1685.0
No. Observations:                 309   AIC:                             3380.
Df Residuals:                     304   BIC:                             3399.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -332.2349     31.853    -10.430      0.0

In [ ]:
test_pred = est_train_significant.predict(df_test)
r2 = r2_score(df_test['Y'],test_pred)
print('OOS R-squared: '+ str(r2))

OOS R-squared: 0.4858788233693758


## Step 8 — Selecting the final model

After the last iterations, the notebook found that the best model possible is the one that uses the variables SEX, BMI, S3 and S5. Even when the model with the variables SEX, BMI, BP and S5 produces a slightly better OSR2, it does not maintain the significance for all the variables.


In [ ]:
est_train_significant = ols(formula = "Y ~ SEX + BMI + S3 + S5", data = df_train).fit()
print(est_train_significant.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.470
Model:                            OLS   Adj. R-squared:                  0.463
Method:                 Least Squares   F-statistic:                     67.34
Date:                Mon, 14 Sep 2026   Prob (F-statistic):           9.47e-41
Time:                        21:24:38   Log-Likelihood:                -1688.3
No. Observations:                 309   AIC:                             3387.
Df Residuals:                     304   BIC:                             3405.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept   -176.6489     42.698     -4.137      0.0

In [ ]:
test_pred = est_train_significant.predict(df_test)
r2 = r2_score(df_test['Y'],test_pred)
print('OOS R-squared: '+ str(r2))

OOS R-squared: 0.48511853284845097


## Technical conclusions

The full model explains **52.4%** of the training variation but reduces its R2 to **47.7%** when using out-of-sample data, while the reduced model achieves **48.5% out-of-sample R²**. That is a useful balance between interpretability and predictive power for a linear baseline.

## Business conclusions

The model is valuable as a transparent benchmark: stakeholders can see which variables are retained and how the prediction is formed. In a healthcare analytics workflow, that transparency can be more useful than a modest performance gain from a black-box model, especially during early model review.

## Limitations and next steps

This is not a clinical decision rule. External validation, robust diagnostics, and comparison with regularized and nonlinear models would be necessary before operational use.
